# Notebook 17: Direct Multiclass Model Evaluation

## Purpose
Evaluate the **dedicated 5-class multiclass model** (`best_multiclass_model.pth`) which directly predicts DR grades 0-4.

This is fundamentally different from the binary→ordinal threshold approach:
- **Binary→Threshold**: Train binary (DR vs No-DR), then map P(DR) to 5 grades via post-hoc thresholds
- **Direct Multiclass**: Train 5-class classifier with CrossEntropyLoss + label smoothing → directly predicts grade

### Model Architecture
- Backbone: EfficientNet-B3 (pretrained)
- Classifier: `nn.Sequential(Dropout(0.4), Linear(1536, 5))`
- Training: Two-phase (frozen backbone → full fine-tune), CrossEntropyLoss with label_smoothing=0.1
- Best validation QWK: **0.859**

In [3]:
import torch
import torch.nn as nn
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import cv2
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    cohen_kappa_score, f1_score, precision_score, recall_score,
    roc_auc_score
)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Setup
DATA_ROOT = Path('.')
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

GRADE_NAMES = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative DR'}
GRADE_SHORT = {0: 'G0', 1: 'G1', 2: 'G2', 3: 'G3', 4: 'G4'}

Device: mps
PyTorch: 2.7.1


## 1. Load Model Architecture & Weights

In [5]:
# Load checkpoint
ckpt = torch.load('models/best_multiclass_merged_balanced_model.pth', map_location='cpu', weights_only=False)

config = ckpt['config']
print('=== Model Configuration ===')
for k, v in config.items():
    print(f'  {k}: {v}')
print(f'\nBest Validation QWK: {ckpt["best_val_qwk"]}')
print(f'Num classes: {ckpt["num_classes"]}')
print(f'Image size: {ckpt["image_size"]}')
print(f'Normalization: mean={ckpt["mean"]}, std={ckpt["std"]}')

FileNotFoundError: [Errno 2] No such file or directory: 'models/best_multiclass_merged_balanced_model.pth'

In [ ]:
# Build model matching checkpoint structure
# The checkpoint uses: classifier = Sequential(Dropout(0.4), Linear(1536, 5))
model = timm.create_model('efficientnet_b3', pretrained=False, num_classes=5)
model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(1536, 5))

# Load weights
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f'Model loaded successfully!')
print(f'Classifier: {model.classifier}')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## 2. Preprocessing Pipeline

In [ ]:
def circle_crop(image):
    height, width = image.shape[:2]
    center_x, center_y = width // 2, height // 2
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    threshold = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)[1]
    contours, _ = cv2.findContours(threshold, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) > 0:
        largest = max(contours, key=cv2.contourArea)
        (cx, cy), radius = cv2.minEnclosingCircle(largest)
        center_x, center_y, radius = int(cx), int(cy), int(radius)
    else:
        radius = min(center_x, center_y)
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.circle(mask, (center_x, center_y), radius, 255, -1)
    cropped = np.zeros_like(image)
    cropped[mask == 255] = image[mask == 255]
    return cropped

def ben_graham_preprocess(image, sigma=30):
    result = np.zeros_like(image, dtype=np.float32)
    for i in range(3):
        ch = image[:, :, i].astype(np.float32)
        blurred = cv2.GaussianBlur(ch, (0, 0), sigma)
        result[:, :, i] = 255 - (255 - ch) * (255 - blurred) / 255
    return np.clip(result, 0, 255).astype(np.uint8)

def apply_clahe(image, clip=2.0):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def resize_with_padding(image, target_size=384):
    h, w = image.shape[:2]
    scale = target_size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_CUBIC)
    canvas = np.zeros((target_size, target_size, 3), dtype=resized.dtype)
    yo, xo = (target_size - nh) // 2, (target_size - nw) // 2
    canvas[yo:yo+nh, xo:xo+nw] = resized
    return canvas

def preprocess_image(image_path, image_size=384):
    """Full preprocessing pipeline matching training."""
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f'Cannot read: {image_path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = circle_crop(img)
    img = ben_graham_preprocess(img)
    img = apply_clahe(img)
    img = resize_with_padding(img, target_size=image_size)
    img = img.astype(np.float32) / 255.0
    return img

def preprocess_to_tensor(image_path, mean, std, image_size=384):
    """Preprocess and normalize to tensor."""
    img = preprocess_image(image_path, image_size)
    # Apply ImageNet normalization
    for c in range(3):
        img[:, :, c] = (img[:, :, c] - mean[c]) / std[c]
    # HWC -> CHW
    tensor = torch.from_numpy(img).permute(2, 0, 1)
    return tensor

print('Preprocessing functions defined.')

## 3. Load Test Data

In [ ]:
# Load test split
test_df = pd.read_csv('splits_aptos/test_split.csv')
print(f'Test set: {len(test_df)} images')
print(f'\nGrade distribution:')
grade_counts = test_df['dr_grade'].value_counts().sort_index()
for grade, count in grade_counts.items():
    pct = count / len(test_df) * 100
    print(f'  Grade {grade} ({GRADE_NAMES[grade]}): {count} ({pct:.1f}%)')

# Verify images exist (use preprocessed if available)
missing = 0
for _, row in test_df.iterrows():
    pp = DATA_ROOT / row['preprocessed_path']
    orig = DATA_ROOT / row['file_path']
    if not pp.exists() and not orig.exists():
        missing += 1
print(f'\nMissing images: {missing}')

## 4. Run Inference (with and without TTA)

In [ ]:
def run_inference(model, test_df, mean, std, image_size=384, device=DEVICE, use_tta=False):
    """
    Run inference on test set.
    TTA: horizontal flip augmentation (average logits of original + flipped).
    """
    all_probs = []
    all_preds = []
    all_logits = []
    errors = []
    
    model.eval()
    with torch.no_grad():
        for idx, row in test_df.iterrows():
            # Try preprocessed first, then original
            pp_path = DATA_ROOT / row['preprocessed_path']
            orig_path = DATA_ROOT / row['file_path']
            img_path = pp_path if pp_path.exists() else orig_path
            
            try:
                tensor = preprocess_to_tensor(str(img_path), mean, std, image_size)
                batch = tensor.unsqueeze(0).to(device)
                
                logits = model(batch)
                
                if use_tta:
                    # Horizontal flip TTA
                    batch_flip = torch.flip(batch, dims=[3])
                    logits_flip = model(batch_flip)
                    logits = (logits + logits_flip) / 2.0
                
                probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
                pred = np.argmax(probs)
                
                all_probs.append(probs)
                all_preds.append(pred)
                all_logits.append(logits.cpu().numpy()[0])
            except Exception as e:
                errors.append((row['image_id'], str(e)))
                all_probs.append(np.zeros(5))
                all_preds.append(-1)
                all_logits.append(np.zeros(5))
            
            if (idx + 1) % 100 == 0:
                print(f'  Processed {idx + 1}/{len(test_df)}...')
    
    if errors:
        print(f'\nWarning: {len(errors)} errors during inference')
        for img_id, err in errors[:5]:
            print(f'  {img_id}: {err}')
    
    return np.array(all_probs), np.array(all_preds), np.array(all_logits)

# Model normalization params
mean = ckpt['mean']  # [0.485, 0.456, 0.406]
std = ckpt['std']    # [0.229, 0.224, 0.225]

print('Running inference WITHOUT TTA...')
probs_noTTA, preds_noTTA, logits_noTTA = run_inference(model, test_df, mean, std, use_tta=False)

print('\nRunning inference WITH TTA (horizontal flip)...')
probs_TTA, preds_TTA, logits_TTA = run_inference(model, test_df, mean, std, use_tta=True)

print(f'\nInference complete: {len(preds_noTTA)} predictions')

## 5. Core Metrics: Accuracy, QWK, F1

In [ ]:
true_labels = test_df['dr_grade'].values

def compute_all_metrics(y_true, y_pred, probs, label=''):
    """Compute comprehensive metrics for multiclass classification."""
    # Filter out error predictions
    mask = y_pred >= 0
    yt, yp, pp = y_true[mask], y_pred[mask], probs[mask]
    
    acc = accuracy_score(yt, yp)
    qwk = cohen_kappa_score(yt, yp, weights='quadratic')
    lwk = cohen_kappa_score(yt, yp, weights='linear')
    f1_macro = f1_score(yt, yp, average='macro', zero_division=0)
    f1_weighted = f1_score(yt, yp, average='weighted', zero_division=0)
    precision_macro = precision_score(yt, yp, average='macro', zero_division=0)
    recall_macro = recall_score(yt, yp, average='macro', zero_division=0)
    
    # Per-class accuracy
    per_class_acc = {}
    for g in range(5):
        mask_g = yt == g
        if mask_g.sum() > 0:
            per_class_acc[g] = (yp[mask_g] == g).mean()
    
    # Adjacent accuracy (within 1 grade)
    adjacent_acc = np.mean(np.abs(yt - yp) <= 1)
    
    metrics = {
        'accuracy': acc,
        'qwk': qwk,
        'lwk': lwk,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'adjacent_accuracy': adjacent_acc,
        'per_class_accuracy': per_class_acc,
        'n_samples': len(yt)
    }
    
    print(f'\n=== {label} ===')
    print(f'  Overall Accuracy:   {acc:.4f} ({acc*100:.1f}%)')
    print(f'  Quadratic Kappa:    {qwk:.4f}')
    print(f'  Linear Kappa:       {lwk:.4f}')
    print(f'  F1 (macro):         {f1_macro:.4f}')
    print(f'  F1 (weighted):      {f1_weighted:.4f}')
    print(f'  Precision (macro):  {precision_macro:.4f}')
    print(f'  Recall (macro):     {recall_macro:.4f}')
    print(f'  Adjacent Accuracy:  {adjacent_acc:.4f} ({adjacent_acc*100:.1f}%)')
    print(f'  Per-class accuracy:')
    for g, a in per_class_acc.items():
        print(f'    Grade {g} ({GRADE_NAMES[g]:20s}): {a:.4f} ({a*100:.1f}%)')
    
    return metrics

metrics_noTTA = compute_all_metrics(true_labels, preds_noTTA, probs_noTTA, 'Multiclass Direct (No TTA)')
metrics_TTA = compute_all_metrics(true_labels, preds_TTA, probs_TTA, 'Multiclass Direct (With TTA)')

## 6. Comparison with Binary→Threshold Approach

In [ ]:
# Load existing binary→ordinal results for comparison
existing_preds = pd.read_csv('evaluation/aptos_comprehensive/test_predictions_full.csv')

# Load ordinal thresholds from evaluation
import glob
eval_json_candidates = glob.glob('evaluation/aptos_comprehensive/**/*eval*.json', recursive=True)
print('Found eval JSONs:', eval_json_candidates)

# Try to load thresholds
ordinal_qwk_tp = None
ordinal_qwk_ens = None

for jf in eval_json_candidates:
    try:
        with open(jf) as f:
            ev = json.load(f)
        if 'ordinal_grading' in ev:
            og = ev['ordinal_grading']
            if 'twophase' in og:
                ordinal_qwk_tp = og['twophase'].get('qwk')
            if 'ensemble' in og:
                ordinal_qwk_ens = og['ensemble'].get('qwk')
            print(f'Loaded ordinal results from {jf}')
            break
    except:
        continue

print(f'\n{"="*70}')
print(f'{"APPROACH COMPARISON":^70}')
print(f'{"="*70}')
print(f'{"Approach":<40} {"QWK":>10} {"Accuracy":>10}')
print(f'{"-"*70}')

if ordinal_qwk_tp is not None:
    print(f'{"Binary→Ordinal (Two-Phase)":<40} {ordinal_qwk_tp:>10.4f} {"N/A":>10}')
if ordinal_qwk_ens is not None:
    print(f'{"Binary→Ordinal (K-Fold Ensemble)":<40} {ordinal_qwk_ens:>10.4f} {"N/A":>10}')

print(f'{"Direct Multiclass (No TTA)":<40} {metrics_noTTA["qwk"]:>10.4f} {metrics_noTTA["accuracy"]:>10.4f}')
print(f'{"Direct Multiclass (With TTA)":<40} {metrics_TTA["qwk"]:>10.4f} {metrics_TTA["accuracy"]:>10.4f}')
print(f'{"="*70}')

if ordinal_qwk_tp is not None:
    improvement = metrics_TTA['qwk'] - ordinal_qwk_tp
    print(f'\nQWK improvement (Direct vs Binary→Ordinal TP): {improvement:+.4f}')

## 7. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, preds, title in [(axes[0], preds_noTTA, 'No TTA'), (axes[1], preds_TTA, 'With TTA')]:
    mask = preds >= 0
    cm = confusion_matrix(true_labels[mask], preds[mask], labels=[0,1,2,3,4])
    
    # Normalize
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                xticklabels=[f'G{i}' for i in range(5)],
                yticklabels=[f'G{i}' for i in range(5)],
                vmin=0, vmax=1)
    
    # Add raw counts
    for i in range(5):
        for j in range(5):
            ax.text(j + 0.5, i + 0.75, f'n={cm[i,j]}', ha='center', va='center',
                   fontsize=7, color='gray')
    
    ax.set_xlabel('Predicted Grade')
    ax.set_ylabel('True Grade')
    ax.set_title(f'Direct Multiclass Model ({title})')

plt.tight_layout()
plt.savefig('evaluation/multiclass_direct_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: evaluation/multiclass_direct_confusion_matrix.png')

## 8. Classification Report

In [ ]:
mask = preds_TTA >= 0
report = classification_report(
    true_labels[mask], preds_TTA[mask],
    labels=[0, 1, 2, 3, 4],
    target_names=[f'G{i}: {GRADE_NAMES[i]}' for i in range(5)],
    digits=4,
    zero_division=0
)
print('=== Classification Report (TTA) ===')
print(report)

## 9. Per-Class Probability Distribution

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for g in range(5):
    ax = axes[g]
    mask = true_labels == g
    if mask.sum() == 0:
        ax.set_title(f'Grade {g} (n=0)')
        continue
    
    # Show predicted probability for each class
    grade_probs = probs_TTA[mask]  # Shape: (n_samples, 5)
    
    # Box plot of probabilities for each predicted class
    bp = ax.boxplot([grade_probs[:, c] for c in range(5)], 
                     labels=[f'G{c}' for c in range(5)],
                     patch_artist=True)
    
    colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    ax.set_title(f'True Grade {g}\n({GRADE_NAMES[g]}, n={mask.sum()})')
    ax.set_ylabel('Predicted Probability' if g == 0 else '')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)

plt.suptitle('Multiclass Model: Predicted Probability Distribution by True Grade', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('evaluation/multiclass_direct_probability_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Error Analysis

In [ ]:
mask = preds_TTA >= 0
yt, yp = true_labels[mask], preds_TTA[mask]

# Error magnitude distribution
errors = yp - yt
abs_errors = np.abs(errors)

print('=== Error Analysis (TTA) ===')
print(f'Mean Absolute Error: {abs_errors.mean():.4f}')
print(f'Median Absolute Error: {np.median(abs_errors):.4f}')
print(f'Max Absolute Error: {abs_errors.max()}')
print(f'\nError distribution:')
for e in range(-4, 5):
    count = (errors == e).sum()
    if count > 0:
        pct = count / len(errors) * 100
        bar = '#' * int(pct)
        direction = 'under' if e < 0 else ('over' if e > 0 else 'correct')
        print(f'  Error {e:+d} ({direction:>7s}): {count:4d} ({pct:5.1f}%) {bar}')

# Clinically significant errors (off by 2+ grades)
severe_errors = abs_errors >= 2
print(f'\nClinically significant errors (|error| >= 2): {severe_errors.sum()} ({severe_errors.mean()*100:.1f}%)')

# Referral accuracy: Grade 0-1 = No Referral, Grade 2-4 = Refer
true_refer = yt >= 2
pred_refer = yp >= 2
referral_acc = (true_refer == pred_refer).mean()
referral_sens = (pred_refer & true_refer).sum() / true_refer.sum() if true_refer.sum() > 0 else 0
referral_spec = (~pred_refer & ~true_refer).sum() / (~true_refer).sum() if (~true_refer).sum() > 0 else 0
print(f'\nReferral Decision (Grade >=2):')
print(f'  Accuracy:    {referral_acc:.4f}')
print(f'  Sensitivity: {referral_sens:.4f}')
print(f'  Specificity: {referral_spec:.4f}')

## 11. Confidence Analysis

In [ ]:
mask = preds_TTA >= 0
yt, yp = true_labels[mask], preds_TTA[mask]
pp = probs_TTA[mask]

# Max probability (confidence) for each prediction
confidences = pp.max(axis=1)
correct = (yt == yp)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confidence histogram: correct vs incorrect
ax = axes[0]
ax.hist(confidences[correct], bins=30, alpha=0.7, label='Correct', color='green', density=True)
ax.hist(confidences[~correct], bins=30, alpha=0.7, label='Incorrect', color='red', density=True)
ax.set_xlabel('Prediction Confidence')
ax.set_ylabel('Density')
ax.set_title('Confidence Distribution')
ax.legend()

# 2. Reliability diagram (calibration)
ax = axes[1]
n_bins = 10
bin_boundaries = np.linspace(0, 1, n_bins + 1)
bin_accs = []
bin_confs = []
bin_counts = []
for i in range(n_bins):
    in_bin = (confidences >= bin_boundaries[i]) & (confidences < bin_boundaries[i+1])
    if in_bin.sum() > 0:
        bin_accs.append(correct[in_bin].mean())
        bin_confs.append(confidences[in_bin].mean())
        bin_counts.append(in_bin.sum())

ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.bar(bin_confs, bin_accs, width=0.08, alpha=0.7, color='steelblue', label='Model')
ax.set_xlabel('Mean Predicted Confidence')
ax.set_ylabel('Fraction of Correct')
ax.set_title('Reliability Diagram')
ax.legend()

# 3. Accuracy vs confidence threshold
ax = axes[2]
thresholds = np.arange(0.1, 1.0, 0.05)
accs_at_thresh = []
coverage_at_thresh = []
for t in thresholds:
    above = confidences >= t
    if above.sum() > 0:
        accs_at_thresh.append(correct[above].mean())
        coverage_at_thresh.append(above.mean())
    else:
        accs_at_thresh.append(np.nan)
        coverage_at_thresh.append(0)

ax.plot(thresholds, accs_at_thresh, 'b-o', markersize=3, label='Accuracy')
ax2 = ax.twinx()
ax2.plot(thresholds, coverage_at_thresh, 'r--s', markersize=3, label='Coverage')
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Accuracy', color='blue')
ax2.set_ylabel('Coverage', color='red')
ax.set_title('Accuracy-Coverage Tradeoff')
ax.legend(loc='lower left')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.savefig('evaluation/multiclass_direct_confidence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Save Results

In [ ]:
# Save predictions CSV
results_df = test_df.copy()
results_df['mc_pred_noTTA'] = preds_noTTA
results_df['mc_pred_TTA'] = preds_TTA
for c in range(5):
    results_df[f'mc_prob_g{c}_noTTA'] = probs_noTTA[:, c]
    results_df[f'mc_prob_g{c}_TTA'] = probs_TTA[:, c]
results_df['mc_confidence_noTTA'] = probs_noTTA.max(axis=1)
results_df['mc_confidence_TTA'] = probs_TTA.max(axis=1)
results_df['mc_correct_noTTA'] = (preds_noTTA == true_labels).astype(int)
results_df['mc_correct_TTA'] = (preds_TTA == true_labels).astype(int)

output_csv = 'evaluation/multiclass_direct_predictions.csv'
results_df.to_csv(output_csv, index=False)
print(f'Saved predictions: {output_csv}')

# Save comprehensive metrics JSON
all_metrics = {
    'model': 'EfficientNet-B3 Direct Multiclass (5-class)',
    'checkpoint': 'models/best_multiclass_model.pth',
    'val_qwk_during_training': float(ckpt['best_val_qwk']),
    'test_set_size': len(test_df),
    'noTTA': {
        'accuracy': metrics_noTTA['accuracy'],
        'qwk': metrics_noTTA['qwk'],
        'lwk': metrics_noTTA['lwk'],
        'f1_macro': metrics_noTTA['f1_macro'],
        'f1_weighted': metrics_noTTA['f1_weighted'],
        'adjacent_accuracy': metrics_noTTA['adjacent_accuracy'],
        'per_class_accuracy': {str(k): v for k, v in metrics_noTTA['per_class_accuracy'].items()}
    },
    'TTA': {
        'accuracy': metrics_TTA['accuracy'],
        'qwk': metrics_TTA['qwk'],
        'lwk': metrics_TTA['lwk'],
        'f1_macro': metrics_TTA['f1_macro'],
        'f1_weighted': metrics_TTA['f1_weighted'],
        'adjacent_accuracy': metrics_TTA['adjacent_accuracy'],
        'per_class_accuracy': {str(k): v for k, v in metrics_TTA['per_class_accuracy'].items()}
    }
}

output_json = 'evaluation/multiclass_direct_eval.json'
with open(output_json, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f'Saved metrics: {output_json}')

print('\n=== SUMMARY ===')
print(f'Direct Multiclass QWK (TTA): {metrics_TTA["qwk"]:.4f}')
print(f'Direct Multiclass Acc (TTA): {metrics_TTA["accuracy"]:.4f}')
print(f'Val QWK during training:     {ckpt["best_val_qwk"]:.4f}')